# 第 05 天：分层回测

> 来自《30 天因子研究计划》第 5 天  
> 主题：分层回测  
> 必做：5 分组回测  
> 选做：10 分组回测  
> 目标产出：分层收益曲线

---

## 0. 今天你要真正学会什么？

第 3 天我们用 IC 判断因子和未来收益是否相关。  
第 4 天我们用 ICIR 判断这种相关是否稳定。  
第 5 天要进入更像投资组合的检验：

> 把股票按因子值分成几组，高分组是否真的跑赢低分组？

这就是分层回测，也叫分组回测、分位数组合回测。

学完以后，你应该能回答：

1. 什么是 5 分组回测？
2. 为什么要看高分组、低分组和多空组合？
3. 什么叫分层收益曲线？
4. 为什么单调性很重要？
5. 10 分组和 5 分组有什么区别？
6. 分层回测和 IC 有什么关系？

一句话版：

> 分层回测是在问：如果真的按因子排序买股票，高因子组的组合表现是否更好？

---

## 1. 先建立直觉：把全班按成绩预估分组

继续第 3 天的班级考试类比。

你先根据“平时作业分”预测学生考试表现。  
然后把学生分成 5 组：


G1：平时作业分最低的 20%
G2
G3
G4
G5：平时作业分最高的 20%


考试后你看每组平均成绩：


G1 < G2 < G3 < G4 < G5


如果真的越高组成绩越好，说明这个预测指标有排序能力。

因子研究里也是一样：


按因子值排序 → 分成 5 组 → 看未来收益 → 画累计收益曲线


---

## 2. 分层回测在检验什么？

IC 是相关性指标。  
分层回测是组合表现指标。

它关心的是：


高因子组是否跑赢低因子组？
分组收益是否大致单调？
多空组合是否长期向上？


典型结果：

| 现象 | 解释 |
| --- | --- |
| G5 明显跑赢 G1 | 因子可能有正向选股能力 |
| G1 明显跑赢 G5 | 因子方向可能反了 |
| 中间组混乱 | 因子排序能力不稳定 |
| 只有最高组好 | 因子可能只在尾部有效 |
| 多空曲线长期向上 | 因子收益可能可组合化 |

分层回测比 IC 更接近投资，因为它开始回答：

> 如果我真的按这个因子买一篮子股票，结果会怎样？

---

## 3. 准备 Python 环境


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

rng = np.random.default_rng(20260705)


如果缺包，可以先安装：


In [ ]:
pip install numpy pandas matplotlib


---

## 4. 构造一份可回测的模拟数据

我们需要一张研究底表：


date | ticker | factor_value | next_1d_ret


这里用 `next_1d_ret` 做日频回测收益：


在 t 日按因子分组
持有到 t+1 日
得到 next_1d_ret


### 4.1 构造日期、股票和因子


In [ ]:
dates = pd.bdate_range("2024-01-02", periods=240)
tickers = [f"Stock_{i:03d}" for i in range(400)]

records = []

for date in dates:
    n = len(tickers)

    # 因子值：截面标准正态
    factor = rng.normal(0, 1, size=n)

    # 市场共同波动
    market_ret = rng.normal(0.0002, 0.008)

    # 个股噪声
    stock_noise = rng.normal(0, 0.018, size=n)

    # 让高因子值股票未来收益略高一点
    next_1d_ret = market_ret + 0.0012 * factor + stock_noise

    for ticker, f, r in zip(tickers, factor, next_1d_ret):
        records.append((date, ticker, f, r))

data = pd.DataFrame(
    records,
    columns=["date", "ticker", "factor_value", "next_1d_ret"]
)

data.head()


这个模拟世界里，因子有轻微正向有效性。  
真实市场里，这个关系通常会更弱、更吵、更不稳定。

---

## 5. 先看一下 IC

分层回测之前，先快速算 Rank IC，看看因子是否有基础排序关系。


In [ ]:
def calc_daily_ic(data: pd.DataFrame, factor_col: str, return_col: str) -> pd.Series:
    ic_values = {}

    for date, group in data.groupby("date"):
        g = group[[factor_col, return_col]].dropna()
        ic_values[date] = g[factor_col].corr(g[return_col], method="spearman")

    return pd.Series(ic_values, name="rank_ic").sort_index()


daily_rank_ic = calc_daily_ic(data, "factor_value", "next_1d_ret")

ic_summary = {
    "mean_rank_ic": daily_rank_ic.mean(),
    "std_rank_ic": daily_rank_ic.std(),
    "icir": daily_rank_ic.mean() / daily_rank_ic.std(),
    "positive_ratio": (daily_rank_ic > 0).mean(),
}

ic_summary


如果 Rank IC 平均为正，分层回测通常也应该呈现高分组更好的倾向。  
如果二者完全矛盾，就要检查分组逻辑、标签方向或收益对齐。

---

## 6. 5 分组回测

### 6.1 写一个分组函数

为了避免重复值导致 `qcut` 报错，我们先用排名再分组。


In [ ]:
def add_quantile_group(
    data: pd.DataFrame,
    factor_col: str = "factor_value",
    n_groups: int = 5,
    group_col: str = "group",
) -> pd.DataFrame:
    """
    按每个日期的因子值分位数分组。

    G1 表示因子值最低组。
    G{n_groups} 表示因子值最高组。
    """
    result = data.copy()

    def group_one_day(s: pd.Series) -> pd.Series:
        ranks = s.rank(method="first")
        group_id = pd.qcut(ranks, q=n_groups, labels=False) + 1
        return group_id.astype(int)

    result[group_col] = (
        result
        .groupby("date")[factor_col]
        .transform(group_one_day)
    )

    return result


grouped_5 = add_quantile_group(data, n_groups=5, group_col="group_5")
grouped_5.head()


### 6.2 检查每组股票数量


In [ ]:
group_count = (
    grouped_5
    .groupby(["date", "group_5"])["ticker"]
    .count()
    .unstack()
)

group_count.head()


每组应该大致一样多。  
如果某天某组特别少，要检查股票池、缺失值和分组逻辑。

### 6.3 计算每组每日收益

最简单的等权组合收益：


组收益 = 组内股票 next_1d_ret 的平均值


In [ ]:
group_ret_5 = (
    grouped_5
    .groupby(["date", "group_5"])["next_1d_ret"]
    .mean()
    .unstack()
    .sort_index()
)

group_ret_5.columns = [f"G{int(c)}" for c in group_ret_5.columns]
group_ret_5.head()


### 6.4 计算累计净值


In [ ]:
group_nav_5 = (1 + group_ret_5).cumprod()
group_nav_5.head()


### 6.5 画 5 分组收益曲线


In [ ]:
group_nav_5.plot(title="5-Group Layered Backtest NAV")
plt.ylabel("NAV")
plt.show()


如果因子有效，通常希望看到：


G5 高因子组 > G4 > G3 > G2 > G1 低因子组


现实里不一定这么完美，但至少高低组应该有明显分化。

---

## 7. 多空组合：高分组减低分组

如果因子是正向的，最简单的多空收益是：


long_short = G5 - G1


In [ ]:
long_short_5 = group_ret_5["G5"] - group_ret_5["G1"]
long_short_nav_5 = (1 + long_short_5).cumprod()

long_short_nav_5.plot(title="5-Group Long-Short NAV: G5 - G1")
plt.ylabel("Long-short NAV")
plt.show()


这条曲线很关键。

如果它长期向上，说明高因子组相对低因子组有稳定超额。  
如果它大起大落，说明因子收益可能不稳定。  
如果它长期向下，因子方向可能反了。

---

## 8. 分组收益是否单调？

除了看曲线，还要看各组平均收益。


In [ ]:
avg_group_ret_5 = group_ret_5.mean().to_frame("mean_daily_ret")
avg_group_ret_5["annualized_ret_approx"] = avg_group_ret_5["mean_daily_ret"] * 252
avg_group_ret_5


画柱状图：


In [ ]:
avg_group_ret_5["annualized_ret_approx"].plot(kind="bar", title="Average Return by Group")
plt.ylabel("Approx annualized return")
plt.show()


一个好看的分层结果通常具有单调性：


G1 < G2 < G3 < G4 < G5


如果只有 G5 高、其他组混乱，也不是完全没用，但说明因子可能主要在尾部有效。

---

## 9. 10 分组回测

10 分组比 5 分组更细。

优点：

- 能更清楚看到尾部分化。
- 更适合观察因子单调性。

缺点：

- 每组股票更少，收益更噪。
- 对股票池数量要求更高。

### 9.1 构造 10 分组


In [ ]:
grouped_10 = add_quantile_group(data, n_groups=10, group_col="group_10")

group_ret_10 = (
    grouped_10
    .groupby(["date", "group_10"])["next_1d_ret"]
    .mean()
    .unstack()
    .sort_index()
)

group_ret_10.columns = [f"G{int(c)}" for c in group_ret_10.columns]
group_nav_10 = (1 + group_ret_10).cumprod()

group_ret_10.head()


### 9.2 画 10 分组曲线


In [ ]:
group_nav_10.plot(title="10-Group Layered Backtest NAV")
plt.ylabel("NAV")
plt.show()


10 条线可能有点拥挤，但能更细地观察：

- G10 是否明显最好？
- G1 是否明显最差？
- 中间组是否大致按顺序排列？

### 9.3 10 分组多空


In [ ]:
long_short_10 = group_ret_10["G10"] - group_ret_10["G1"]
long_short_nav_10 = (1 + long_short_10).cumprod()

long_short_nav_10.plot(title="10-Group Long-Short NAV: G10 - G1")
plt.ylabel("Long-short NAV")
plt.show()


---

## 10. 回测指标汇总

写一个简单的绩效函数。


In [ ]:
def performance_summary(ret: pd.Series, periods_per_year: int = 252) -> pd.Series:
    clean = ret.dropna()
    nav = (1 + clean).cumprod()

    ann_ret = clean.mean() * periods_per_year
    ann_vol = clean.std() * np.sqrt(periods_per_year)
    sharpe = ann_ret / ann_vol if ann_vol != 0 else np.nan
    max_drawdown = (nav / nav.cummax() - 1).min()

    return pd.Series({
        "ann_ret_approx": ann_ret,
        "ann_vol": ann_vol,
        "sharpe_approx": sharpe,
        "max_drawdown": max_drawdown,
        "win_rate": (clean > 0).mean(),
    })


summary_5 = pd.DataFrame({
    col: performance_summary(group_ret_5[col])
    for col in group_ret_5.columns
}).T

summary_5.loc["G5-G1"] = performance_summary(long_short_5)
summary_5


注意：这里用的是简化年化方法，适合教学。  
真实回测要考虑调仓频率、交易成本、停牌、涨跌停、权重约束和复利精确计算。

---

## 11. 目标产出：一键生成分层收益曲线

把今天的流程封装成函数。


In [ ]:
def layered_backtest(
    data: pd.DataFrame,
    factor_col: str = "factor_value",
    return_col: str = "next_1d_ret",
    n_groups: int = 5,
) -> dict:
    group_col = f"group_{n_groups}"
    grouped = add_quantile_group(data, factor_col=factor_col, n_groups=n_groups, group_col=group_col)

    group_ret = (
        grouped
        .groupby(["date", group_col])[return_col]
        .mean()
        .unstack()
        .sort_index()
    )
    group_ret.columns = [f"G{int(c)}" for c in group_ret.columns]

    group_nav = (1 + group_ret).cumprod()

    low_group = "G1"
    high_group = f"G{n_groups}"
    long_short_ret = group_ret[high_group] - group_ret[low_group]
    long_short_nav = (1 + long_short_ret).cumprod()

    summary = pd.DataFrame({
        col: performance_summary(group_ret[col])
        for col in group_ret.columns
    }).T
    summary.loc[f"{high_group}-{low_group}"] = performance_summary(long_short_ret)

    return {
        "grouped": grouped,
        "group_ret": group_ret,
        "group_nav": group_nav,
        "long_short_ret": long_short_ret,
        "long_short_nav": long_short_nav,
        "summary": summary,
    }


bt5 = layered_backtest(data, n_groups=5)
bt10 = layered_backtest(data, n_groups=10)


画图：


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 9), sharex=True)

bt5["group_nav"].plot(ax=axes[0], title="5-Group Layered NAV")
axes[0].set_ylabel("NAV")

bt5["long_short_nav"].plot(ax=axes[1], title="Long-Short NAV: G5 - G1", color="black")
axes[1].set_ylabel("Long-short NAV")

plt.tight_layout()
plt.show()

bt5["summary"]


这就是第 5 天的目标产出：  
分层收益曲线 + 多空收益曲线 + 分组绩效表。

---

## 12. 分层回测和 IC 的关系

IC 和分层回测看的是同一个因子的不同侧面。

| 工具 | 关注点 |
| --- | --- |
| IC | 因子值和未来收益的截面相关 |
| ICIR | IC 是否稳定 |
| 分层回测 | 按因子排序买组合能不能分化 |
| 多空组合 | 高因子组相对低因子组是否有超额 |

一个理想因子通常会同时表现为：

- Rank IC 均值为正。
- ICIR 较稳定。
- 高分组净值高于低分组。
- 分组收益大致单调。
- 多空组合长期向上。

如果 IC 好但分层回测不好，可能原因有：

- 因子只影响极少数股票。
- 收益被极端值驱动。
- 分组权重和 IC 计算口径不一致。
- 交易成本吞掉收益。
- 标签和回测收益对齐方式不同。

---

## 13. 分层回测质量检查清单

### 13.1 分组数量是否适合股票池？

股票太少时，不适合分太多组。

### 13.2 每组股票数量是否均衡？

每组数量异常，先查缺失值和分组逻辑。

### 13.3 因子方向是否正确？

如果 G1 长期跑赢 G5，可能因子方向反了。

### 13.4 是否有单调性？

理想情况是从低组到高组收益逐步提高。

### 13.5 多空曲线是否长期向上？

高低组差异是否能稳定积累。

### 13.6 是否考虑交易成本？

今天为了学习暂时忽略，但真实研究必须考虑。

### 13.7 是否有未来函数？

分组使用的因子必须在调仓时点可知。

### 13.8 是否过度依赖某一阶段？

分年度、分市场状态检查很重要。

---

## 14. 加入一个简单交易成本

今天先用一个非常简化的成本模型：


每天多空组合扣除固定成本 0.02%


In [ ]:
cost_per_day = 0.0002
long_short_5_after_cost = long_short_5 - cost_per_day
long_short_nav_5_after_cost = (1 + long_short_5_after_cost).cumprod()

pd.DataFrame({
    "before_cost": long_short_nav_5,
    "after_cost": long_short_nav_5_after_cost,
}).plot(title="Long-Short NAV Before and After Simple Cost")

plt.ylabel("NAV")
plt.show()


真实交易成本更复杂，至少包括：

- 手续费
- 印花税
- 滑点
- 冲击成本
- 换手率
- 买卖停牌和涨跌停限制

但这个简化例子能提醒你：

> 因子收益如果很薄，交易成本可能直接把它吃掉。

---

## 15. 今天的知识图谱


In [ ]:
mindmap
  root((分层回测))
    输入
      date
      ticker
      factor_value
      next_return
    分组
      5分组
      10分组
      qcut
      每日截面排序
    组收益
      等权平均
      高分组
      低分组
      中间组
    曲线
      分层净值
      多空净值
      成本后净值
    判断
      单调性
      高低组分化
      多空长期向上
      回撤
    风险点
      未来函数
      交易成本
      分组数量过多
      样本不足
      因子方向错误


文本版：


分层回测
├── 输入
│   ├── 因子值
│   └── 下一期收益
├── 分组
│   ├── 5 分组
│   └── 10 分组
├── 收益
│   ├── 组内等权收益
│   ├── 高分组收益
│   └── 多空收益
├── 图表
│   ├── 分层收益曲线
│   └── 多空净值曲线
└── 检查
    ├── 单调性
    ├── 方向
    ├── 成本
    └── 未来函数


---

## 16. 初学者最容易踩的 9 个坑

### 坑 1：全样本排序后分组

必须按每个日期单独分组。不能把所有日期混在一起排序。

### 坑 2：因子方向反了

如果低分组长期跑赢高分组，先检查指标方向。

### 坑 3：看见高分组好就忽略单调性

只有最高组好，说明因子可能只在尾部有效，不一定适合全排序组合。

### 坑 4：分组太多

股票池太小时，10 分组会让每组股票太少，噪声变大。

### 坑 5：忽略交易成本

高换手因子可能纸面收益好，实盘收益差。

### 坑 6：用未来不可知的因子分组

因子值必须在调仓时点可知。

### 坑 7：混淆 forward return 和实际回测收益

分层均值检验可以用未来 20 日收益；连续净值曲线最好明确调仓和持有规则。

### 坑 8：只看总图，不看分阶段

分年、分市场状态看，才能知道因子是否稳定。

### 坑 9：忽略行业和市值暴露

高分组跑赢可能只是因为偏小盘、偏某行业，后续需要中性化。

---

## 17. 今天的动手作业

### 作业 A：解释分层回测

用自己的话回答：

1. 为什么分层回测要按日期分组？
2. G5 跑赢 G1 说明什么？
3. 分组收益单调性为什么重要？

### 作业 B：运行 5 分组回测

运行本文代码，保存：

- 5 分组净值曲线
- G5-G1 多空曲线
- 分组绩效表

### 作业 C：运行 10 分组回测

比较 5 分组和 10 分组：

- 哪个图更清晰？
- 哪个更容易噪声大？
- G10 和 G1 差异是否更明显？

### 作业 D：改变因子强度

把这行：


In [ ]:
next_1d_ret = market_ret + 0.0012 * factor + stock_noise


改成：


In [ ]:
next_1d_ret = market_ret + 0.0003 * factor + stock_noise
next_1d_ret = market_ret - 0.0012 * factor + stock_noise


观察分层曲线如何变化。

### 作业 E：加入交易成本

把每日成本改成：


In [ ]:
0.0001
0.0005
0.0010


看多空净值曲线还能否向上。

---

## 18. 自测题

### 题 1

5 分组回测中的 G1 和 G5 分别代表什么？

答案：G1 是因子值最低的 20% 股票，G5 是因子值最高的 20% 股票。

### 题 2

为什么分组必须每天重新做？

答案：因为每天股票的因子值和股票池都会变化，分层回测检验的是每个日期截面的排序能力。

### 题 3

如果 G1 长期跑赢 G5，可能说明什么？

答案：因子方向可能反了，或者因子代表的是负向暴露。

### 题 4

多空组合 G5-G1 的含义是什么？

答案：做多高因子组，做空低因子组，观察高低组之间的相对收益。

### 题 5

分层回测好看是否等于可以实盘？

答案：不等于。还需要交易成本、换手率、容量、风险控制、停牌涨跌停、样本外验证等检查。

---

## 19. 今日复盘模板


第 05 天复盘：分层回测

1. 我今天理解的分层回测：

2. 5 分组结果是否单调：

3. G5-G1 多空曲线是否向上：

4. 10 分组相比 5 分组的变化：

5. 加入交易成本后的变化：

6. 我认为这个因子还能继续研究吗：

7. 明天学习价值因子前，我需要准备：


---

## 20. 明天预告：价值因子 1

前 5 天我们已经完成了因子研究的基础检验链条：


因子值
  ↓
未来收益标签
  ↓
IC
  ↓
ICIR
  ↓
分层回测


第 6 天开始会进入具体因子：价值因子。  
你会开始研究 PE、PB、EV/EBITDA 等指标。

---

## 21. 一句话收尾

IC 告诉你因子有没有排序关系，分层回测告诉你这种排序关系放进组合里像不像钱。

> 一个因子能不能继续研究，分层收益曲线通常会给出非常直观的第一印象。

---

## 22. 仅供学习的提醒

本文所有示例使用模拟数据，仅用于解释分层回测方法，不构成任何投资建议。真实策略研究需要考虑复权、停牌、退市、涨跌停、交易成本、换手率、行业市值暴露、容量和样本外验证。

---

# 统一高质量增强模块

> 本增强模块用于把第 05 天课程统一提升到第 1-2 天那种“能直接学习、能直接运行、能直接复盘”的密度。前面的正文保留；下面是更完整的学习版。

## A. 今日任务重新聚焦

- 主题：分层回测
- 必做：5分组回测
- 选做：10分组回测
- 目标产出：分层收益曲线

今天真正要练成的不是“知道一个名词”，而是能把这个主题放进完整因子研究流水线：


原始数据
  ↓
因子构造
  ↓
预处理和对齐
  ↓
IC / ICIR / 分层回测
  ↓
形成可复用模块


你学习时可以一直问自己三句话：

1. 这个因子在经济含义上解释什么？
2. 这个因子在代码里如何被严格计算？
3. 这个因子是否真的经得起检验，而不是只在故事里成立？

## B. 一个更生动的直觉案例

如果按因子从低到高把股票分成五组，高分组真的长期跑赢低分组吗？这比一个相关系数更接近投资组合。

这个例子背后的关键直觉是：

> 分层回测把因子排序翻译成组合表现。

因子研究不是把金融名词翻译成代码，而是把一个投资假设拆成可以被验证、被复现、被质疑的实验。

## C. 今日知识骨架


分层回测
├── 输入数据
│   ├── 行情 / 财务 / 行业 / 市值等基础字段
│   └── 明确每个字段在当时是否可得
├── 因子定义
│   ├── 写清楚公式
│   ├── 写清楚方向
│   └── 写清楚缺失和异常值处理
├── 因子检验
│   ├── Rank IC
│   ├── ICIR
│   └── 分层回测
└── 目标产出
    └── 分层收益曲线


## D. 完整 Python 实验

下面这段代码是一个自包含实验。你可以单独复制到 Notebook 里运行。它的目的不是模拟真实市场，而是把今天主题的计算口径、方向、检查方法串起来。


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(105)
dates = pd.bdate_range("2024-01-02", periods=120)
tickers = [f"S{i:03d}" for i in range(300)]
rows = []
for d in dates:
    factor = rng.normal(0, 1, len(tickers))
    next_ret = 0.001 * factor + rng.normal(0, 0.018, len(tickers))
    rows.extend(zip([d] * len(tickers), tickers, factor, next_ret))
df = pd.DataFrame(rows, columns=["date", "ticker", "factor", "next_ret"])
df["group"] = df.groupby("date")["factor"].transform(
    lambda s: pd.qcut(s.rank(method="first"), 5, labels=False) + 1
)
group_ret = df.groupby(["date", "group"])["next_ret"].mean().unstack()
group_ret.columns = [f"G{int(c)}" for c in group_ret.columns]
nav = (1 + group_ret).cumprod()
long_short = group_ret["G5"] - group_ret["G1"]
print(group_ret.mean().round(5))
print("long_short_ann_ret_approx:", round(long_short.mean() * 252, 4))


## E. 产出验收标准

完成今天课程后，你的 `分层收益曲线` 至少应该满足：

1. 字段命名清晰，能看出日期、股票、因子值和标签含义。
2. 因子方向明确：值越大到底代表越好、越便宜、越强，还是越低风险。
3. 缺失值和异常值有处理口径，不把未知伪装成 0。
4. 至少有一段可重复运行的 Python 实验验证核心逻辑。
5. 能用 IC、ICIR 或分层回测中的至少一种方法做初步检查。
6. 能解释这个因子在真实研究里可能失效的原因。

如果这些检查没有过，不要急着进入下一天。因子研究里很多错误不是模型问题，而是最开始的口径、方向、对齐、缺失值处理出了问题。

## F. 常见坑深挖

### 坑 1：只记公式，不检查数据可得时点

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 2：因子方向写反，却直接进入 IC 和回测

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 3：把模拟数据里的漂亮结果当成真实市场规律

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 4：忽略缺失值、极端值和样本边界

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 5：只看单一指标，不做交叉验证

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 6：没有把目标产出封装成可复用函数

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。

## G. 强化练习

### 作业 A

用自己的话写出 `分层回测` 的一句话定义，并标明它属于收益、风险、估值、质量、技术、流动性还是预处理模块。
### 作业 B

运行完整实验代码，记录输出结果，并解释每一列结果的金融含义。
### 作业 C

故意把因子方向取反，再重新计算结果，观察 IC 或分组表现如何变化。
### 作业 D

加入 5% 缺失值或 1% 极端值，测试你的处理逻辑是否仍然稳健。
### 作业 E

把今天的 `分层收益曲线` 保存成一个可以被后续课程调用的函数或表格。

## H. 面试式自测

### 问：这个主题在因子研究流水线里处于哪一步？

答：它对应 `分层收益曲线`，用于把原始数据转成后续 IC、ICIR、分层回测或多因子合成可以使用的中间产物。
### 问：最容易出现未来函数的地方在哪里？

答：通常出现在使用未来才披露的数据、未来价格、未来收益标签错位，或把全样本统计量用于历史截面。
### 问：为什么不能只看一个漂亮结果？

答：因为单次结果可能来自样本偶然、极端值、行业暴露、市值暴露或参数过拟合，需要多角度验证。
### 问：如何判断今天产出的模块可以进入下一步？

答：至少通过字段检查、方向检查、缺失异常检查、抽样手工验证和一个简单统计检验。

## I. 今日复盘模板


第 05 天复盘：分层回测

1. 今天我能用一句话解释的核心概念：

2. 今天最重要的公式：

3. 代码里最容易写错的地方：

4. 我检查因子方向的方法：

5. 我检查缺失值和异常值的方法：

6. 如果把这个模块放进真实研究，我还缺什么数据：

7. 今天留下的一个问题：


## J. 和下一课的连接

下一课会继续沿着这条链路推进：前一天产出的字段或模块，会成为后一天检验、扩展或组合的输入。学习时不要把每天割裂开；真正的因子研究是一条流水线。

---

## K. 学习提醒

这一份课程仍然是教学材料，示例数据是模拟数据。真实研究需要处理真实数据源、可得时点、复权、停牌、交易成本、行业和市值暴露、样本外验证。
